# Perturbation Flux Comparison

For one heliostat, apply **two manually defined perturbation sets** and compare
the resulting flux images on sun positions drawn from the synthetic dataset.

**Workflow:**
1. Edit `HELIOSTAT_ID`, `SPLIT`, and display settings in **Cell 2** (Config)
2. Edit the two perturbation dicts in **Cell 4** (the only cell to change per run)
3. Run all cells — the comparison grid appears at the end

In [ ]:
import pathlib
import sys
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

warnings.filterwarnings("ignore")

# Path setup -- notebook lives in src/perturbation_visualizer/
_nb   = pathlib.Path(globals().get("__vsc_ipynb_file__", pathlib.Path().resolve() / "x")).parent
_src  = _nb.parent   # .../src
_base = _src.parent  # .../master-thesis
sys.path.insert(0, str(_src))

from artist.scenario.scenario import Scenario
from artist.util import get_device, set_logger_config

from utils.synth_data import (
    _forward_pass,
    apply_perturbations,
    reset_perturbations,
    SyntheticDatasetParser,
)

from nb_utils import normalize_for_display, bitmap_centroid

set_logger_config()
import logging
logging.getLogger().setLevel(logging.WARNING)

print("Imports OK")

In [ ]:
# ============================================================
#  CONFIG
# ============================================================

HELIOSTAT_ID       = "AC33"             # AC33  AC36  AG33  AO34  AW36  BE35
SPLIT              = "test"             # "train" / "val" / "test"
DATASET_SPLIT_TYPE = "azimuth_dataset"  # "azimuth_dataset" or "balanced_dataset"
N_DISPLAY          = 10                 # samples shown in the comparison grid
SURFACE_PTS        = 25                 # surface points per facet side (25x25)
DISPLAY_RAYS       = 50                 # rays per surface point for each forward pass

# Derived paths
SCENARIO_PATH = _base / "scenarios" / "one_heliostat_scenarios" / HELIOSTAT_ID / "scenario.h5"
DATASET_DIR   = _base / "datasets" / "synthetic" / DATASET_SPLIT_TYPE / "dataset"
SPLIT_DIR     = DATASET_DIR / SPLIT / HELIOSTAT_ID

print(f"Heliostat : {HELIOSTAT_ID}")
print(f"Scenario  : {SCENARIO_PATH}")
print(f"Dataset   : {DATASET_DIR / SPLIT}")

## Perturbation parameters

Edit the two dicts in **Cell 4**. All shapes are for **N = 1** (single heliostat).
Set a component to `0.0` to leave it unperturbed.

| Key | Shape | Unit | Typical bound (Wortberg 2025, Table 5.3) | Meaning |
|---|---|---|---|---|
| `translation` | [1, 9] | m | ±0.05 | Joint + concentrator XYZ translations (3+3+3) |
| `rotation` | [1, 4] | rad | ±0.005 | Joint tilt deviations (4 joints) |
| `actuator_angle` | [1, 2] | rad | ±0.005 | Actuator initial angles a_i (optimised) |
| `actuator_stroke` | [1, 2] | m | ±0.005 | Actuator initial stroke lengths b_i (frozen) |
| `actuator_offset` | [1, 2] | m | ±0.005 | Actuator offsets c_i (optimised) |
| `base_position` | [1, 3] | m | ±0.05 | Base position shift [east, north, up] |

In [ ]:
# ============================================================
#  PERTURBATION DEFINITIONS  <-- edit this cell
#  All values in SI units (metres / radians).
#  Zero means "no deviation from the clean heliostat".
# ============================================================

PERT_A_LABEL = "Perturbation A"
PERT_A = {
    # 9 values: joint_1 (x,y,z), joint_2 (x,y,z), concentrator (x,y,z)  [m]
    "translation":     torch.tensor([[0.0, 0.0, 0.0,  0.0, 0.0, 0.0,  0.0, 0.0, 0.0]]),
    # 4 joint tilt angles  [rad]
    "rotation":        torch.tensor([[0.0, 0.0, 0.0, 0.0]]),
    # 2 actuator initial angles  a_i  [rad]
    "actuator_angle":  torch.tensor([[0.0, 0.0]]),
    # 2 actuator initial stroke lengths  b_i  [m]
    "actuator_stroke": torch.tensor([[0.0, 0.0]]),
    # 2 actuator offsets  c_i  [m]
    "actuator_offset": torch.tensor([[0.0, 0.0]]),
    # base position shift  [east, north, up]  [m]
    "base_position":   torch.tensor([[0.0, 0.0, 0.0]]),
}

PERT_B_LABEL = "Perturbation B"
PERT_B = {
    "translation":     torch.tensor([[0.0, 0.0, 0.0,  0.0, 0.0, 0.0,  0.0, 0.0, 0.0]]),
    "rotation":        torch.tensor([[0.0, 0.0, 0.0, 0.0]]),
    "actuator_angle":  torch.tensor([[0.0, 0.0]]),
    "actuator_stroke": torch.tensor([[0.0, 0.0]]),
    "actuator_offset": torch.tensor([[0.0, 0.0]]),
    "base_position":   torch.tensor([[0.0, 0.0, 0.0]]),
}

print(f"A: {PERT_A_LABEL}")
print(f"B: {PERT_B_LABEL}")

In [ ]:
# ============================================================
#  Load scenario
# ============================================================

device = get_device()
print(f"Device: {device}")

with h5py.File(SCENARIO_PATH, "r") as f:
    scenario = Scenario.load_scenario_from_hdf5(
        scenario_file=f,
        device=device,
        number_of_surface_points_per_facet=torch.tensor([SURFACE_PTS, SURFACE_PTS]),
    )

heliostat_group = scenario.heliostat_field.heliostat_groups[0]
kinematic       = heliostat_group.kinematics
scenario.set_number_of_rays(DISPLAY_RAYS)

print(f"Scenario loaded: {SCENARIO_PATH.name}")
print(f"Rays per surface point: {DISPLAY_RAYS}")

In [ ]:
# ============================================================
#  Load sun positions and GT flux from the synthetic dataset
# ============================================================

n_on_disk = sum(1 for p in SPLIT_DIR.iterdir() if p.is_dir() and p.name.isdigit())
parser    = SyntheticDatasetParser(DATASET_DIR / SPLIT)
mapping   = [(HELIOSTAT_ID, list(range(n_on_disk)), list(range(n_on_disk)))]

gt_flux, gt_centroids, rays, motor_pos, active_mask, target_mask = (
    parser.parse_data_for_reconstruction(
        heliostat_data_mapping=mapping,
        heliostat_group=heliostat_group,
        scenario=scenario,
        device=device,
    )
)
# gt_flux:      [N, H, W]  normalized [0, 1]  (from PNG files on disk)
# gt_centroids: [N, 4]     ENU focal spots
# rays:         [N, 3]     incident ray directions
# target_mask:  [N]        target area index per sample

n_total = len(rays)
n_disp  = min(N_DISPLAY, n_total)

# Take the first n_disp samples (deterministic)
disp_rays   = rays[:n_disp]
disp_target = target_mask[:n_disp]
disp_gt     = gt_flux[:n_disp]
disp_active = torch.tensor([n_disp], device=device, dtype=torch.long)

print(f"Loaded {n_total} samples from {SPLIT_DIR}")
print(f"Showing first {n_disp} samples in the comparison grid")

In [ ]:
# ============================================================
#  Forward pass -- Perturbation A
# ============================================================

snap_a = apply_perturbations(kinematic, PERT_A, device)
# Read total base-position deviation after perturbation (handles any pre-existing offset)
_bpd_a  = kinematic._base_position_deviation.detach().clone()  # [1, 3]
cents_a, flux_a = _forward_pass(
    scenario, heliostat_group,
    disp_rays, disp_active, disp_target,
    _bpd_a, device,
)
reset_perturbations(kinematic, snap_a)
# flux_a:  [n_disp, H, W]  physical intensity units
# cents_a: [n_disp, 4]     ENU centroids

print(f"Perturbation A done  |  flux range [{flux_a.min():.3e}, {flux_a.max():.3e}]")

In [ ]:
# ============================================================
#  Forward pass -- Perturbation B
# ============================================================

snap_b = apply_perturbations(kinematic, PERT_B, device)
_bpd_b  = kinematic._base_position_deviation.detach().clone()  # [1, 3]
cents_b, flux_b = _forward_pass(
    scenario, heliostat_group,
    disp_rays, disp_active, disp_target,
    _bpd_b, device,
)
reset_perturbations(kinematic, snap_b)

print(f"Perturbation B done  |  flux range [{flux_b.min():.3e}, {flux_b.max():.3e}]")

In [ ]:
# ============================================================
#  Flux comparison grid
#  Rows = samples, Columns = [Dataset GT | Pert A | Pert B]
# ============================================================

# _pix_centroid and _for_display are now in nb_utils (imported in cell 1 as
# bitmap_centroid and normalize_for_display).
_for_display = normalize_for_display


col_titles = ["Dataset GT", PERT_A_LABEL, PERT_B_LABEL]
fig, axes = plt.subplots(
    n_disp, 3,
    figsize=(9, 2.8 * n_disp),
    gridspec_kw={"hspace": 0.08, "wspace": 0.04},
)
if n_disp == 1:
    axes = axes[np.newaxis, :]

for si in range(n_disp):
    gt_img = _for_display(disp_gt[si])
    a_img  = _for_display(flux_a[si], ref=disp_gt[si])
    b_img  = _for_display(flux_b[si], ref=disp_gt[si])

    for col_i, img in enumerate([gt_img, a_img, b_img]):
        ax = axes[si, col_i]
        ax.imshow(img, cmap="inferno", vmin=0, vmax=1, origin="upper")
        cx, cy = bitmap_centroid(torch.from_numpy(np.asarray(img, dtype=np.float32)))
        if cx is not None:
            ax.plot(cx, cy, "+", color="white", markersize=9, markeredgewidth=1.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if si == 0:
            ax.set_title(col_titles[col_i], fontsize=10)
        if col_i == 0:
            ax.set_ylabel(f"#{si}", fontsize=8, rotation=0, labelpad=24, va="center")

fig.suptitle(
    f"{HELIOSTAT_ID}  --  flux comparison ({SPLIT} set)\n"
    f"col 1: dataset GT  |  col 2: {PERT_A_LABEL}  |  col 3: {PERT_B_LABEL}",
    fontsize=10, y=1.01,
)
plt.show()

In [ ]:
# ============================================================
#  Centroid statistics (ENU, metres)
# ============================================================

header = (
    f"{'#':>4}  "
    f"{'A-east':>9} {'A-north':>9} {'A-up':>9}  "
    f"{'B-east':>9} {'B-north':>9} {'B-up':>9}  "
    f"{'|A-B| m':>9}"
)
print(header)
print("-" * len(header))

dists = []
for si in range(n_disp):
    ca = cents_a[si, :3].cpu().numpy()
    cb = cents_b[si, :3].cpu().numpy()
    d  = float(np.linalg.norm(ca - cb))
    dists.append(d)
    print(
        f"{si:>4}  "
        f"{ca[0]:>+9.4f} {ca[1]:>+9.4f} {ca[2]:>+9.4f}  "
        f"{cb[0]:>+9.4f} {cb[1]:>+9.4f} {cb[2]:>+9.4f}  "
        f"{d:>9.4f}"
    )

print("-" * len(header))
print(
    f"Mean |A-B| = {np.mean(dists):.4f} m  |  "
    f"Median = {np.median(dists):.4f} m  |  "
    f"Max = {np.max(dists):.4f} m"
)